# AMI-CPSD v1.0: Exploratory Data Analysis (EDA)
## Advanced Metering Infrastructure Cyber-Physical Synthetic Digital Twin

This notebook performs exploratory analysis on the **AMI-CPSD v1.0** dataset, inspecting cross-domain feature distributions, grid topology, temporal profiles, and cyber-physical attack characteristics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### 1. Load Dataset & Metadata

In [ ]:
data_dir = Path("../datasets")
meta_dir = Path("../metadata")

df_train = pd.read_csv(data_dir / "train.csv", nrows=50000)
with open(meta_dir / "metadata.json") as f:
    metadata = json.load(f)

print(f"Loaded {len(df_train):,} training observations.")
print(f"Total features: {len(df_train.columns)}")
display(df_train.head())

### 2. Attack Class Distribution

In [ ]:
plt.figure(figsize=(12, 5))
ax = sns.countplot(data=df_train, x="attack_category", palette="viridis")
plt.title("Attack Class Frequencies in Training Dataset Split", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Observation Count")
plt.tight_layout()
plt.show()

### 3. Electrical Power Physics Verification: $P = V \cdot I \cdot \cos\phi / 1000$

In [ ]:
# Calculate expected active power P_calc
df_train["p_calc"] = (df_train["voltage"] * df_train["current"] * df_train["power_factor"]) / 1000.0
df_train["p_diff"] = np.abs(df_train["p_calc"] - df_train["active_power"])

plt.figure(figsize=(9, 5))
sns.boxplot(data=df_train, x="attack_category", y="p_diff", palette="rocket")
plt.title("Physical Power Residual |P_calc - P_measured| Across Attack Types", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Physics Violation Residual (kW)")
plt.tight_layout()
plt.show()

### 4. Cross-Domain Feature Correlation Matrix

In [ ]:
num_cols = ["voltage", "current", "active_power", "packet_rate", "packet_delay", "cpu_usage", "consensus_latency", "anomaly_score", "entropy"]
plt.figure(figsize=(9, 7))
sns.heatmap(df_train[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Cross-Domain Cyber-Physical Feature Correlation Heatmap", fontsize=12, fontweight="bold")
plt.show()